<a href="https://colab.research.google.com/github/kuds/reinforce-tactics/blob/claude/kaggle-environments-compatibility-K4RyW/notebooks/kaggle_environments_smoke_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reinforce Tactics — Kaggle Environments Smoke Test

End-to-end validation that the **Reinforce Tactics** Kaggle Environments adapter is wired up correctly.

This notebook:
1. Installs `kaggle-environments` and clones this branch.
2. Stages the adapter into `kaggle_environments/envs/reinforce_tactics/` — the same path `kaggle/kaggle-environments` PR #673 deploys to.
3. Runs a series of `make()` / `reset()` / `run()` smoke tests.
4. Executes both pytest suites (the 21 functional tests and the 90 comprehensive internal tests).

If every cell completes without errors, you have a working setup that mirrors what the upstream PR will produce.

## 1. Setup

Install `kaggle-environments`, clone this branch, then copy the adapter into the `kaggle_environments/envs/` directory before any first import. The import-time directory scan is how `make("reinforce_tactics")` discovers the environment.

In [ ]:
import importlib.util, shutil, subprocess

# Install kaggle-environments first — but DO NOT import it yet.
subprocess.run(['pip', 'install', '-q', 'kaggle-environments', 'pytest'], check=True)

# Find the install path without importing the module.
ke_dir = importlib.util.find_spec('kaggle_environments').submodule_search_locations[0]
print('kaggle_environments install path:', ke_dir)

In [ ]:
# Clone this branch.
REPO_DIR = '/tmp/reinforce-tactics'
BRANCH = 'claude/kaggle-environments-compatibility-K4RyW'

subprocess.run(['rm', '-rf', REPO_DIR], check=False)
subprocess.run([
    'git', 'clone', '--depth', '1', '-b', BRANCH,
    'https://github.com/kuds/reinforce-tactics.git', REPO_DIR,
], check=True)

# Stage the adapter into kaggle_environments/envs/reinforce_tactics/.
target = f'{ke_dir}/envs/reinforce_tactics'
shutil.rmtree(target, ignore_errors=True)
shutil.copytree(f'{REPO_DIR}/reinforcetactics/kaggle', target)
print('Adapter staged at:', target)

## 2. Smoke tests via the public API

Now that the adapter is on disk, importing `kaggle_environments` will register it and `make("reinforce_tactics")` will return a working `Environment`.

In [ ]:
from kaggle_environments import make, evaluate

env = make(
    'reinforce_tactics',
    configuration={'mapName': 'beginner', 'episodeSteps': 30, 'mapSeed': 42},
    debug=False,
)
state = env.reset()

print('Number of agent slots :', len(state))
print('Player 0 status      :', state[0]['status'])
print('Player 1 status      :', state[1]['status'])
print('Starting gold        :', state[0]['observation']['gold'])
print('Map name (config)    :', env.configuration.mapName)
print('Board size           :', len(state[0]['observation']['board']),
      'x', len(state[0]['observation']['board'][0]))

structures = state[0]['observation']['structures']
print('Structure count      :', len(structures))
print('Structure types      :', sorted({s["type"] for s in structures}))

In [ ]:
# ASCII render of the initial board
print(env.render(mode='ansi'))

In [ ]:
# Run a full episode: random agent vs the strategic simple_bot.
result = env.run(['random', 'simple_bot'])
final = result[-1]

print(f'Turns simulated      : {len(result) - 1}')
print(f'Final status (P0/P1) : {final[0].status} / {final[1].status}')
print(f'Final reward (P0/P1) : {final[0].reward} / {final[1].reward}')

if final[0].reward == final[1].reward == 0:
    print('Outcome              : Draw or step-cap reached')
elif final[0].reward > final[1].reward:
    print('Outcome              : random agent won')
else:
    print('Outcome              : simple_bot won')

## 3. Configuration variations

Spot-check that the configurable knobs all work: the three built-in maps, random generation, fog of war, custom starting gold, and the `evaluate()` helper.

In [ ]:
# Each of the three built-in maps should reset cleanly to a 20x20 board.
for name in ['beginner', 'crossroads', 'tower_rush']:
    e = make('reinforce_tactics', configuration={'mapName': name})
    s = e.reset()
    rows = len(s[0]['observation']['board'])
    cols = len(s[0]['observation']['board'][0])
    print(f'  {name:<12} -> {rows}x{cols} board, status={s[0]["status"]}')

In [ ]:
# Random map (mapName='') and fog of war.
rand_env = make('reinforce_tactics', configuration={'mapName': '', 'mapSeed': 7})
rand_state = rand_env.reset()
ocean_count = sum(row.count('o') for row in rand_state[0]['observation']['board'])
print(f'Random map ocean tiles: {ocean_count}')

fog_env = make('reinforce_tactics', configuration={'fogOfWar': True, 'mapSeed': 42})
fog_state = fog_env.reset()
print(f'Fog of war units list type: {type(fog_state[0]["observation"]["units"]).__name__}')

rich_env = make('reinforce_tactics', configuration={'startingGold': 1000, 'mapSeed': 42})
rich_state = rich_env.reset()
print(f'Custom starting gold      : {rich_state[0]["observation"]["gold"]}')

In [ ]:
# evaluate(): play several episodes and collect rewards.
rewards = evaluate(
    'reinforce_tactics',
    ['random', 'random'],
    num_episodes=3,
    configuration={'mapSeed': 42, 'episodeSteps': 20},
)
print('Per-episode rewards (each row sums to 0):')
for i, r in enumerate(rewards):
    print(f'  Episode {i+1}: P0={r[0]}, P1={r[1]}')

## 4. Run the unit-test suites

Two test suites ship in the staging repo:

* **`tests/test_kaggle_env_make.py`** — 21 functional tests that exercise the env through `kaggle_environments.make()`, mirroring the test file that will live at `tests/envs/reinforce_tactics/test_reinforce_tactics.py` once PR #673 merges upstream.
* **`tests/test_kaggle_env.py`** — 90 unit tests against the adapter internals (interpreter, serialisation, action execution, win/draw conditions, built-in agents). Staging-only; not shipped upstream.

Both run from the cloned repo. We pass `--override-ini=addopts=` to neutralise this project's `pytest-cov` config (which expects the full ML stack to be importable).

In [ ]:
# Suite 1: 21 functional tests via make().
import os
result = subprocess.run(
    ['pytest', 'tests/test_kaggle_env_make.py', '-v',
     '--override-ini=addopts=', '--no-header', '-p', 'no:cacheprovider'],
    cwd=REPO_DIR,
    env={**os.environ, 'PYTHONPATH': REPO_DIR},
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise SystemExit(f'pytest failed with exit code {result.returncode}')
print('Suite 1: PASS')

In [ ]:
# Suite 2: 90 comprehensive internal tests.
result = subprocess.run(
    ['pytest', 'tests/test_kaggle_env.py',
     '--override-ini=addopts=', '--no-header', '-p', 'no:cacheprovider', '-q'],
    cwd=REPO_DIR,
    env={**os.environ, 'PYTHONPATH': REPO_DIR},
    capture_output=True,
    text=True,
)
print(result.stdout[-2000:])  # tail in case the verbose output is long
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise SystemExit(f'pytest failed with exit code {result.returncode}')
print('Suite 2: PASS')

## Done

If every cell ran without errors, the staged adapter behaves correctly through `kaggle_environments.make()` and both test suites are green. This is the same shape PR #673 produces once merged.

**Next steps:**
* Browse `kaggle_environments/envs/reinforce_tactics/README.md` (now staged in your runtime) for the action / observation reference.
* Submit your own agent: write a function `agent(observation, configuration) -> list[dict]` and pass it to `env.run([my_agent, 'random'])`.
* See `notebooks/bot_tournament.ipynb` for higher-level evaluation tooling.